In [4]:
!jq '. | length' ../data/dataset.json

jq: error: Could not open file ../data/dataset.json: No such file or directory


In [6]:

!jq ".[$(jq 'keys[]' ../data/corpus_embs/dataset.json | shuf -n 1)]" ../data/corpus_embs/dataset.json

{
  "id": "pk_14776",
  "author_reputation": 61,
  "version": 2,
  "fork_count": 1,
  "likes": 3,
  "upvotes": 2,
  "downvotes": 0,
  "views": 17,
  "uses": 4,
  "created_at": "2024-10-16T14:07:10Z",
  "title": "side quest ideas",
  "content": "give me like 10 side quest ideas that dont involve killing stuff. im tierd of every quest being go here kill X enemies",
  "category": "gaming",
  "subcategory": "quest-writing",
  "tags": [
    "side-quests",
    "ideas",
    "variety",
    "non-combat"
  ],
  "has_placeholders": false,
  "placeholders": [],
  "difficulty": "beginner",
  "language": "en",
  "target_model": "llama-3.3-70b"
}


In [22]:

!jq ".[$(jq 'keys[]' ../data/benchmarks/json/queries.json | shuf -n 1)]" ../data/benchmarks/json/queries.json

{
  "query_id": "q_08",
  "query_text": "Generate B2B sales discovery questions to qualify leads using the BANT framework.",
  "expected_prompt_id": "pk_02268",
  "query_type": "Task-Oriented / Generative"
}


In [17]:
from pathlib import Path
import numpy as np 
import json
import matplotlib.pyplot as plt
import seaborn as sns
from arrowspace import ArrowSpaceBuilder
import time

BASE           = Path("~/code_base/leaf_semantic_engine").expanduser()
dataset_json = BASE / "data/corpus_embs/dataset.json"
nomic_raw_path = BASE / "data/corpus_embs/nomic_embs_raw"
qn_dir         = BASE / "data/benchmarks/query_embs_nomic/raw"
DATASET_PATH   = BASE / "data/corpus_embs/dataset.json"
Q_IDX_PATH     = qn_dir / "queries_index.json"

# ── Fixed config: 768d only, tau=0.75 (established best from prior eval)
CORPUS_PATH = nomic_raw_path / "embeddings_nomic_structured_768d_raw.npy"
Q_EMB_PATH  = qn_dir / "queries_emb_768.npy"

TAU = 0.75
GRAPH_PARAMS = {"eps": 0.625, "k": 20, "topk": 10, "p": 2.0, "sigma": None} # from optimazzation routine in embs_eval.ipynb


In [9]:
with open(DATASET_PATH) as f:
    dataset = json.load(f)

In [20]:
corpus_embs = np.load(CORPUS_PATH).astype(np.float64)
query_embs  = np.load(Q_EMB_PATH).astype(np.float64)

In [12]:
print(f"Corpus embs shape: {corpus_embs.shape}")
print(f"Query embs shape: {query_embs.shape}")

Corpus embs shape: (20000, 768)
Query embs shape: (10, 768)


In [15]:
print(f"document example from dataset: {dataset[0]}\nembedding example from dataset: {corpus_embs[0][:10]}")

document example from dataset: {'id': 'pk_03138', 'author_reputation': 65, 'version': 2, 'fork_count': 19, 'likes': 478, 'upvotes': 277, 'downvotes': 6, 'views': 2770, 'uses': 776, 'created_at': '2024-08-25T00:47:27Z', 'title': 'Complete Brand Identity Brief', 'content': 'Create a comprehensive brand identity brief document for a new sustainable fashion startup. Include all essential sections: Executive Summary, Brand Purpose and Mission, Target Audience Persona (demographics, psychographics, behaviors), Brand Positioning statement, Brand Personality (with detailed trait descriptions), Competitive Analysis summary, Core Brand Values, Brand Promise, Tone of Voice guidelines (with examples for different scenarios), and Visual Direction overview. Format this as a professional document a design agency would deliver to stakeholders.', 'category': 'brand-identity', 'subcategory': 'strategic-brief', 'tags': ['brand-brief', 'sustainable-fashion', 'strategy', 'identity'], 'has_placeholders': Fa

# Build arrowspace 

In [21]:
t0 = time.perf_counter()
aspace, gl = ArrowSpaceBuilder().build(GRAPH_PARAMS, corpus_embs)
t1 = time.perf_counter()
print(f"Time to build arrow space: {t1 - t0:.2f} s")

Time to build arrow space: 90.90 s


In [28]:
for query in query_embs:
    result = aspace.search(query, gl, TAU)
    for idx, score in result:
        print(f"Score: {score:.4f}, Document: {dataset[idx]}")

[(6011, 0.8577637982633497), (4, 0.8217256171820172), (9223, 0.8144085102934353), (19745, 0.8106212294938662), (9765, 0.8104885884228574), (5357, 0.8069774448163699), (13495, 0.8062165140876337), (5668, 0.8011970714204232), (6856, 0.8007662257098201), (554, 0.8006641625704018)]
[(10909, 0.8448103198530368), (10605, 0.8275168012743034), (11517, 0.817824693774297), (14, 0.8172453840961889), (10011, 0.8136714520848058), (15650, 0.8122097431562885), (10743, 0.8111168903285746), (4718, 0.8056224557558133), (14261, 0.8055505810893356), (12884, 0.8020460909638054)]
[(21, 0.8167374287201143), (13495, 0.7176048155801299), (9283, 0.7051361169589914), (6713, 0.696939394122541), (18583, 0.693163983935389), (2190, 0.6928473427224214), (16202, 0.6843457816457091), (7969, 0.6810854633732415), (16272, 0.6796089372228094), (11254, 0.678492037626244)]
[(14263, 0.7400514397973441), (1261, 0.7373180714775021), (9406, 0.7367633543486871), (9885, 0.7353929808999862), (3859, 0.7250668358036652), (2464, 0.724